# Week 6 -- Function 1: GP + UCB Exploitation (2D)

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 10
DATA_PATH_X  = '../data/function_1/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_1/initial_outputs.npy'

weekly_log = [
    ([0.591837, 0.591837], 0.00028209052469858225),        # W1: UCB beta=2.5
    ([0.0, 1.0], 0.0),                                     # W2: boundary dead
    ([1.0, 1.0], 1.5141828121885904e-192),                  # W3: corner dead
    ([0.639955, 0.673176], -0.012808060176723465),           # W4: NN grad ascent -- negative
    ([0.001256, 0.00102], 6.006625024649171e-247),           # W5: GP EI+SVM -- dead corner
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Already up-to-date.
X: (15, 2) | Y: (15,)
Week 6 | 15 total observations (10 initial + 5 added)


In [2]:
# Cell 3: Load data and inspect -- Function 1: Radiation Contamination Detection (2D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by magnitude -- signal strength matters, not sign
pairs = sorted(zip(Y.tolist(), X.tolist()), key=lambda p: abs(p[0]), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 72)
print('  All observations (sorted descending by |Y|)')
print('=' * 72)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.6f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.6e}{marker}')
print('=' * 72)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.8f}' for v in best_X)
print(f'\n  Best |Y*| = {abs(best_Y):.6e}  (Y* = {best_Y:.6e})')
print(f'  Best X*   = [{best_x_str}]')

Input  shape : (15, 2)
Output shape : (15,)

  All observations (sorted descending by |Y|)
  [ 1]  X = [0.639955, 0.673176]   Y = -1.280806e-02  <-- best
  [ 2]  X = [0.650114, 0.681526]   Y = -3.606063e-03
  [ 3]  X = [0.591837, 0.591837]   Y = +2.820905e-04
  [ 4]  X = [0.731024, 0.733000]   Y = +7.710875e-16
  [ 5]  X = [0.683418, 0.861057]   Y = +2.535001e-40
  [ 6]  X = [0.574329, 0.879898]   Y = +1.033078e-46
  [ 7]  X = [0.883890, 0.582254]   Y = +6.229856e-48
  [ 8]  X = [0.410437, 0.147554]   Y = -2.159249e-54
  [ 9]  X = [0.319404, 0.762959]   Y = +1.322677e-79
  [10]  X = [0.082507, 0.403488]   Y = +3.606771e-81
  [11]  X = [0.312691, 0.078723]   Y = -2.089093e-91
  [12]  X = [0.840353, 0.264732]   Y = +3.341771e-124
  [13]  X = [1.000000, 1.000000]   Y = +1.514183e-192
  [14]  X = [0.001256, 0.001020]   Y = +6.006625e-247
  [15]  X = [0.000000, 1.000000]   Y = +0.000000e+00

  Best |Y*| = 1.280806e-02  (Y* = -1.280806e-02)
  Best X*   = [0.63995500, 0.67317600]


In [3]:
# Cell 4: Fit GP
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

# Log-transform to handle extreme scale differences across observations
Y_fit = np.log(np.abs(Y) + 1e-300)
kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
print('  Sanity check at best known X*:')
print(f'    X*                     = [{best_X[0]:.6f}, {best_X[1]:.6f}]')
print(f'    GP predicted mean      = {pred_mean[0]:.4f}  (log-space)')
print(f'    Actual log(|Y*|)       = {actual_log:.4f}  (log-space)')
print(f'    GP predicted std       = {pred_std[0]:.8f}')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.1)
  Log-marginal-likelihood : -604598.8265
  Sanity check at best known X*:
    X*                     = [0.639955, 0.673176]
    GP predicted mean      = -4.3577  (log-space)
    Actual log(|Y*|)       = -4.3577  (log-space)
    GP predicted std       = 0.00001000


In [4]:
# Cell 5: GP UCB Acquisition -- focused on hot zone, exploitation mode

# Focused grid around the region where signal exists
x1 = np.linspace(0.55, 0.75, 100)
x2 = np.linspace(0.55, 0.75, 100)
xx1, xx2 = np.meshgrid(x1, x2)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel()])

gp_mean, gp_std = gp.predict(X_grid, return_std=True)

# beta=1.0: exploitation -- we've explored enough, signal is only in this region
beta = 1.0
ucb_scores = gp_mean + beta * gp_std

best_idx = np.argmax(ucb_scores)
next_x = X_grid[best_idx]
portal_string = f'{next_x[0]:.6f}-{next_x[1]:.6f}'

print('=' * 62)
print('  GP UCB Acquisition (beta=1.0, exploitation)')
print('=' * 62)
print(f'  Search grid  : [0.55, 0.75]^2  (10,000 points)')
print(f'  Beta         : {beta}  (exploitation)')
next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Next query   : [{next_str}]')
print(f'  UCB score    : {ucb_scores[best_idx]:.4f}')
print(f'  GP mean      : {gp_mean[best_idx]:.4f}')
print(f'  GP std       : {gp_std[best_idx]:.6f}')
print('=' * 62)

  GP UCB Acquisition (beta=1.0, exploitation)
  Search grid  : [0.55, 0.75]^2  (10,000 points)
  Beta         : 1.0  (exploitation)
  Next query   : [0.646970, 0.644949]
  UCB score    : -2.0726
  GP mean      : -2.2938
  GP std       : 0.221233


In [5]:
# Cell 6: Sanity check -- ensure query stays in hot zone

HOT_LO, HOT_HI = 0.55, 0.75
dist_to_best = np.linalg.norm(next_x - best_X)
in_hot_zone  = all(HOT_LO <= v <= HOT_HI for v in next_x)

print('=' * 62)
print('  Sanity Check')
print('=' * 62)
print(f'  Proposed query    : [{next_x[0]:.6f}, {next_x[1]:.6f}]')
best_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'  Best known X*     : [{best_str}]')
print(f'  Distance to X*    : {dist_to_best:.6f}')
print(f'  In [0.55,0.75]^2  : {in_hot_zone}')
print()

if dist_to_best > 0.3:
    print('  WARNING: query is far from best known point (> 0.3). Consider revising.')
else:
    print('  OK: query is close to best known point.')

if not in_hot_zone:
    print('  WARNING: query is outside [0.55, 0.75]^2 hot zone. Dead corner risk!')
else:
    print('  OK: query is inside the hot zone.')
print('=' * 62)

  Sanity Check
  Proposed query    : [0.646970, 0.644949]
  Best known X*     : [0.639955, 0.673176]
  Distance to X*    : 0.029085
  In [0.55,0.75]^2  : True

  OK: query is close to best known point.
  OK: query is inside the hot zone.


In [6]:
print('=' * 72)
print('  SUMMARY -- Week 6 Bayesian Optimisation')
print('=' * 72)
print('  Function        : 1 -- Radiation Contamination Detection (2D)')
print('  Objective       : Maximise |Y|')
print(f'  Method          : GP UCB (beta=1.0, exploitation, focused grid)')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X*         : [{best_x_s}]')
print(f'  Best Y*         : {best_Y:.6e}  (|Y*| = {abs(best_Y):.6e})')
print()
print(f'  Next query      : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 6 Bayesian Optimisation
  Function        : 1 -- Radiation Contamination Detection (2D)
  Objective       : Maximise |Y|
  Method          : GP UCB (beta=1.0, exploitation, focused grid)

  Best X*         : [0.639955, 0.673176]
  Best Y*         : -1.280806e-02  (|Y*| = 1.280806e-02)

  Next query      : [0.646970, 0.644949]

  Portal submission string:
  >>> 0.646970-0.644949 <<<


## Week 6 -- Reflection

**Function**: F1 -- Radiation Contamination Detection (2D)

### Strategy change from Week 5
- Removed SVM filter: 15 observations, 13 near-zero -- classifier has no signal to learn from.
- Sorting now uses `|Y|` (magnitude), not raw `Y` -- a strong negative reading is a real signal.
- Grid narrowed to `[0.55, 0.75]^2`: both meaningful readings (W1, W4) live here.
- Beta reduced from 3.5 to 1.0: switched from exploration to exploitation.

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 7
*(fill in after reflecting on result)*